# XLM-R sentence rewrite detector on Kaggle

Execution-only entry point for `xlmr_sentence_rewrite`. The research design and evaluation live in
`notebooks/04_sentence_rewrite_experiments.ipynb` and `scripts/run_rewrite_experiments.py`.

The notebook fine-tunes `FacebookAI/xlm-roberta-base` for one epoch to classify single sentences
as rewritten or retained, using labels derived locally from verbatim overlap between training fusion
documents and their linked human documents (family fold 0 of development, excluding families linked
to the holdout). It then scores every eligible sentence of `train.csv` and `test.csv` once.

Inputs: the original competition data and the private `hw3-sentence-transformer-bundle` dataset
(sentence keys, labels, counts and the sentence splitter; no competition text). Internet is needed
only for pinned packages and open model weights.

`MODE = "preflight"` verifies inputs, sentence alignment and GPU without loading the model.
`MODE = "train"` trains and scores; Run All then trains. Outputs are written to
`/kaggle/working/sentence_scores.parquet` and `sentence_scores.json`. Nothing is submitted.

In [ ]:
MODE = "preflight"
SEED = 2026
MODEL_ID = "FacebookAI/xlm-roberta-base"
REVISION = "e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"
MAX_LENGTH = 96
EPOCHS = 1
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
TRAIN_BATCH_SIZE = 64
INFERENCE_BATCH_SIZE = 512
MONITOR_FRACTIONS = [0.25, 0.5, 0.75, 1.0]
INPUT_ROOT = "/kaggle/input"
WORKING_ROOT = "/kaggle/working"
if MODE not in {"preflight", "train"}:
    raise ValueError("MODE must be preflight or train")
MODE

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("/kaggle").exists():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "polars==2.1.0",
            "transformers==5.24.0",
            "tokenizers==0.23.3",
        ],
        check=True,
    )

In [ ]:
import hashlib
import json
import logging
import platform
import shutil
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import polars as pl
import torch
from IPython.display import display
from sklearn.metrics import roc_auc_score

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger("xlmr_sentence")
WORKING = Path(WORKING_ROOT)
WORKING.mkdir(parents=True, exist_ok=True)


def sha256(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


bundle_root = next(Path(INPUT_ROOT).rglob("bundle_metadata.json")).parent
bundle = json.loads((bundle_root / "bundle_metadata.json").read_text())
bundle_files = {name: sha256(bundle_root / name) for name in bundle["files"]}
if bundle_files != bundle["files"]:
    raise ValueError("Bundle files differ from their recorded fingerprints")
inputs = {}
for name, key in [("train.csv", "source_sha256"), ("test.csv", "test_sha256")]:
    matches = [path for path in Path(INPUT_ROOT).rglob(name) if sha256(path) == bundle[key]]
    if len(matches) != 1:
        raise ValueError(f"Expected one original {name} with the recorded fingerprint")
    inputs[name] = matches[0]
package = WORKING / "src/hm3"
package.mkdir(parents=True, exist_ok=True)
(package / "__init__.py").write_text("")
for name in ["families.py", "validation.py"]:
    shutil.copy(bundle_root / name, package / name)
sys.path.insert(0, str(WORKING / "src"))
from hm3.families import split_sentences  # noqa: E402

environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    **{name: version(name) for name in ["polars", "transformers", "tokenizers", "scikit-learn"]},
    "free_disk_gib": round(shutil.disk_usage(WORKING).free / 1024**3, 1),
}
display(
    environment,
    {key: bundle[key] for key in ["training_sentences", "monitor_sentences", "scored_sentences"]},
)

In [ ]:
start = time.perf_counter()
documents = pl.concat(
    [
        pl.read_csv(inputs[name], schema_overrides={"id": pl.String}, columns=["id", "text"])
        for name in ["train.csv", "test.csv"]
    ]
)
sentences = split_sentences(documents, bundle["minimum_sentence_words"]).select(
    "id", "position", "sentence"
)
del documents
counts = sentences.group_by("id").agg(pl.len().alias("sentences")).sort("id")
expected = pl.read_parquet(bundle_root / "sentence_counts.parquet").sort("id")
if not counts.equals(expected):
    raise ValueError("Sentence splitting differs from the local recipe")
labels = {
    name: pl.read_parquet(bundle_root / f"{name}_labels.parquet").join(
        sentences, on=["id", "position"], how="left", validate="1:1"
    )
    for name in ["transformer", "monitor"]
}
if any(frame["sentence"].null_count() for frame in labels.values()):
    raise ValueError("Labelled sentence keys are missing from the split")
display(
    {
        "sentences": sentences.height,
        "split_seconds": round(time.perf_counter() - start, 1),
        "training_rewritten_share": labels["transformer"]["is_rewritten"].mean(),
    }
)

In [ ]:
from transformers import (  # noqa: E402
    AutoModelForSequenceClassification,
    AutoTokenizer,
    PreTrainedTokenizerBase,
    get_linear_schedule_with_warmup,
)


class AutocastClassifier(torch.nn.Module):
    # DataParallel replicas run in threads, so autocast must be entered inside forward.
    def __init__(self, model: torch.nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        with torch.autocast("cuda", dtype=torch.float16):
            return self.model(input_ids=input_ids, attention_mask=attention_mask).logits.float()


def score(
    network: torch.nn.Module,
    tokenizer: PreTrainedTokenizerBase,
    texts: list[str],
    batch_size: int,
) -> np.ndarray:
    # Sorting by length reduces padding; scores are written back in input order.
    order = np.argsort([len(text) for text in texts], kind="stable")
    scores = np.empty(len(texts), dtype=np.float32)
    network.eval()
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            batch = order[start : start + batch_size]
            encoded = tokenizer(
                [texts[i] for i in batch],
                truncation=True,
                max_length=MAX_LENGTH,
                padding=True,
                return_tensors="pt",
            ).to("cuda")
            logits = network(encoded["input_ids"], encoded["attention_mask"])
            scores[batch] = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
    return scores


if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required")
display({"mode": MODE, "will_train": MODE == "train"})

In [ ]:
history = []
model_directory = WORKING / "xlmr_sentence_model"
if MODE == "train":
    torch.manual_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, revision=REVISION, num_labels=2
    ).to("cuda")
    network = AutocastClassifier(base)
    if torch.cuda.device_count() > 1:
        network = torch.nn.DataParallel(network)
    texts = labels["transformer"]["sentence"].to_list()
    targets = torch.tensor(labels["transformer"]["is_rewritten"].cast(pl.Int64).to_numpy())
    monitor_texts = labels["monitor"]["sentence"].to_list()
    monitor_targets = labels["monitor"]["is_rewritten"].to_numpy()
    steps_per_epoch = int(np.ceil(len(texts) / TRAIN_BATCH_SIZE))
    total_steps = steps_per_epoch * EPOCHS
    optimizer = torch.optim.AdamW(base.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(WARMUP_RATIO * total_steps), total_steps
    )
    scaler = torch.amp.GradScaler("cuda")
    checkpoints = {int(fraction * total_steps) for fraction in MONITOR_FRACTIONS}
    generator = np.random.default_rng(SEED)
    step, start = 0, time.perf_counter()
    for epoch in range(EPOCHS):
        order = generator.permutation(len(texts))
        network.train()
        for offset in range(0, len(texts), TRAIN_BATCH_SIZE):
            batch = order[offset : offset + TRAIN_BATCH_SIZE]
            encoded = tokenizer(
                [texts[i] for i in batch],
                truncation=True,
                max_length=MAX_LENGTH,
                padding=True,
                return_tensors="pt",
            ).to("cuda")
            logits = network(encoded["input_ids"], encoded["attention_mask"])
            loss = torch.nn.functional.cross_entropy(logits, targets[batch].to("cuda"))
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(base.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            step += 1
            if step % 500 == 0:
                logger.info("step %s/%s loss %.4f", step, total_steps, loss.item())
            if step in checkpoints:
                auc = roc_auc_score(
                    monitor_targets, score(network, tokenizer, monitor_texts, INFERENCE_BATCH_SIZE)
                )
                history.append(
                    {"step": step, "monitor_auc": auc, "seconds": time.perf_counter() - start}
                )
                logger.info("monitor AUC at step %s: %.4f", step, auc)
                network.train()
    training_seconds = time.perf_counter() - start
    base.save_pretrained(model_directory)
    tokenizer.save_pretrained(model_directory)
    display(pl.DataFrame(history), {"training_seconds": round(training_seconds, 1)})
else:
    display({"training": "skipped in preflight"})

In [ ]:
if MODE == "train":
    start = time.perf_counter()
    all_scores = score(network, tokenizer, sentences["sentence"].to_list(), INFERENCE_BATCH_SIZE)
    scoring_seconds = time.perf_counter() - start
    table = sentences.select("id", "position").with_columns(pl.Series("score", all_scores))
    temporary = WORKING / "sentence_scores.tmp.parquet"
    table.write_parquet(temporary)
    temporary.replace(WORKING / "sentence_scores.parquet")
    record = {
        "model_id": MODEL_ID,
        "revision": REVISION,
        "bundle_files": bundle_files,
        "environment": environment,
        "settings": {
            "max_length": MAX_LENGTH,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "seed": SEED,
        },
        "history": history,
        "monitor_auc": history[-1]["monitor_auc"],
        "training_seconds": training_seconds,
        "scoring_seconds": scoring_seconds,
        "scored_sentences": table.height,
        "scores_sha256": sha256(WORKING / "sentence_scores.parquet"),
    }
    (WORKING / "sentence_scores.json").write_text(json.dumps(record, indent=2))
    display(
        {
            key: record[key]
            for key in ["monitor_auc", "training_seconds", "scoring_seconds", "scored_sentences"]
        }
    )
else:
    display({"scoring": "skipped in preflight"})